# HW3 — Two retrievers and a model: what RAG actually sends

**Name:**
**Student ID:**
**Group:**
**Repository:**

## AI tool disclosure
State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.


---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.


In [1]:
import math
import re

D1 = "Large language models can generate text and answer questions."
D2 = "Retrieval-Augmented Generation retrieves relevant documents before generating an answer."
D3 = "Embeddings represent text as numerical vectors."
query = "How does RAG retrieve information?"

def tokenize(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s-]+", " ", text)
    return [tok for tok in re.split(r"[\s-]+", text) if tok]

DOCS = {"D1": D1, "D2": D2, "D3": D3}
query_terms = tokenize(query)
print("Query terms:", query_terms)

doc_tokens = {k: tokenize(v) for k, v in DOCS.items()}
print("Document tokens:")
for k, toks in doc_tokens.items():
    print(k, toks, "len=", len(toks))

len_by_doc = {k: len(v) for k, v in doc_tokens.items()}
avgdl = sum(len_by_doc.values()) / len(len_by_doc)
print("avgdl =", avgdl)

terms = sorted(set(query_terms))
print("\nTF / DF / IDF")
for term in terms:
    tf = {doc: doc_tokens[doc].count(term) for doc in DOCS}
    df = sum(1 for doc in DOCS if term in doc_tokens[doc])
    idf = math.log((len(DOCS) - df + 0.5) / (df + 0.5))
    print(term, tf, "df=", df, "idf=", idf)

k1 = 1.5
b = 0.75
N = len(DOCS)

def bm25_score(term, doc_id):
    f = doc_tokens[doc_id].count(term)
    df = sum(1 for d in DOCS if term in doc_tokens[d])
    idf = math.log((N - df + 0.5) / (df + 0.5))
    denom = f + k1 * (1 - b + b * len_by_doc[doc_id] / avgdl)
    if denom == 0:
        return 0.0
    return idf * f * (k1 + 1) / denom

scores = {}
for doc_id in DOCS:
    total = 0.0
    contribs = []
    for term in query_terms:
        if term not in doc_tokens[doc_id]:
            continue
        score = bm25_score(term, doc_id)
        total += score
        contribs.append((term, score))
    scores[doc_id] = total
    print(f"{doc_id}: score = {total:.6f}; contributors = {contribs}")

print("\nFinal ranking:")
for doc_id, score in sorted(scores.items(), key=lambda kv: kv[1], reverse=True):
    print(doc_id, score)


Query terms: ['how', 'does', 'rag', 'retrieve', 'information']
Document tokens:
D1 ['large', 'language', 'models', 'can', 'generate', 'text', 'and', 'answer', 'questions'] len= 9
D2 ['retrieval', 'augmented', 'generation', 'retrieves', 'relevant', 'documents', 'before', 'generating', 'an', 'answer'] len= 10
D3 ['embeddings', 'represent', 'text', 'as', 'numerical', 'vectors'] len= 6
avgdl = 8.333333333333334

TF / DF / IDF
does {'D1': 0, 'D2': 0, 'D3': 0} df= 0 idf= 1.9459101490553132
how {'D1': 0, 'D2': 0, 'D3': 0} df= 0 idf= 1.9459101490553132
information {'D1': 0, 'D2': 0, 'D3': 0} df= 0 idf= 1.9459101490553132
rag {'D1': 0, 'D2': 0, 'D3': 0} df= 0 idf= 1.9459101490553132
retrieve {'D1': 0, 'D2': 0, 'D3': 0} df= 0 idf= 1.9459101490553132
D1: score = 0.000000; contributors = []
D2: score = 0.000000; contributors = []
D3: score = 0.000000; contributors = []

Final ranking:
D1 0.0
D2 0.0
D3 0.0


## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | | |
| D2 | | |
| D3 | | |

**avgdl =**

**Stemmed query terms:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| | | | | | |

**BM25 of each document**, worked:

>

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | | |
| D2 | | |
| D3 | | |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

>


In [2]:
stem_map = {
    "retrieve": "retriev",
    "retrieves": "retriev",
    "retrieval": "retriev",
    "generate": "gener",
    "generating": "gener",
    "generation": "gener",
    "information": "inform",
    "does": "doe",
}

stemmed_doc_tokens = {doc_id: [stem_map.get(tok, tok) for tok in tokenize(text)] for doc_id, text in DOCS.items()}
stemmed_query = [stem_map.get(tok, tok) for tok in query_terms]
print("Stemmed query terms:", stemmed_query)
print("Stemmed document tokens:")
for k, toks in stemmed_doc_tokens.items():
    print(k, toks, "len=", len(toks))

stem_len_by_doc = {k: len(v) for k, v in stemmed_doc_tokens.items()}
stem_avgdl = sum(stem_len_by_doc.values()) / len(stem_len_by_doc)
print("stem avgdl =", stem_avgdl)

stem_terms = sorted(set(stemmed_query))
for term in stem_terms:
    tf = {doc: stemmed_doc_tokens[doc].count(term) for doc in DOCS}
    df = sum(1 for doc in DOCS if term in stemmed_doc_tokens[doc])
    idf = math.log((len(DOCS) - df + 0.5) / (df + 0.5))
    print(term, tf, "df=", df, "idf=", idf)

stem_scores = {}
for doc_id in DOCS:
    total = 0.0
    for term in stemmed_query:
        if term not in stemmed_doc_tokens[doc_id]:
            continue
        f = stemmed_doc_tokens[doc_id].count(term)
        df = sum(1 for d in DOCS if term in stemmed_doc_tokens[d])
        idf = math.log((len(DOCS) - df + 0.5) / (df + 0.5))
        denom = f + 1.5 * (1 - 0.75 + 0.75 * stem_len_by_doc[doc_id] / stem_avgdl)
        if denom:
            total += idf * f * (1.5 + 1) / denom
    stem_scores[doc_id] = total
    print(f"{doc_id}: stem score = {total:.6f}")

print("\nFinal stem ranking:")
for doc_id, score in sorted(stem_scores.items(), key=lambda kv: kv[1], reverse=True):
    print(doc_id, score)


Stemmed query terms: ['how', 'doe', 'rag', 'retriev', 'inform']
Stemmed document tokens:
D1 ['large', 'language', 'models', 'can', 'gener', 'text', 'and', 'answer', 'questions'] len= 9
D2 ['retriev', 'augmented', 'gener', 'retriev', 'relevant', 'documents', 'before', 'gener', 'an', 'answer'] len= 10
D3 ['embeddings', 'represent', 'text', 'as', 'numerical', 'vectors'] len= 6
stem avgdl = 8.333333333333334
doe {'D1': 0, 'D2': 0, 'D3': 0} df= 0 idf= 1.9459101490553132
how {'D1': 0, 'D2': 0, 'D3': 0} df= 0 idf= 1.9459101490553132
inform {'D1': 0, 'D2': 0, 'D3': 0} df= 0 idf= 1.9459101490553132
rag {'D1': 0, 'D2': 0, 'D3': 0} df= 0 idf= 1.9459101490553132
retriev {'D1': 0, 'D2': 2, 'D3': 0} df= 1 idf= 0.5108256237659907
D1: stem score = 0.000000
D2: stem score = 0.685672
D3: stem score = 0.000000

Final stem ranking:
D2 0.6856719782093835
D1 0.0
D3 0.0


## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | | | | |
| Query · D2 | | | | |
| Query · D3 | | | | |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | | |
| D2 | | |
| D3 | | |

**Which document is most similar to the query?**

>

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

>

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

>


In [3]:
Query = np.array([0.8, 0.2, 0.5])
D1 = np.array([0.7, 0.1, 0.4])
D2 = np.array([0.9, 0.3, 0.6])
D3 = np.array([0.1, 0.8, 0.2])
docs = {"D1": D1, "D2": D2, "D3": D3}
pairs = [("D1", D1), ("D2", D2), ("D3", D3)]
for name, vec in pairs:
    dot = float(np.dot(Query, vec))
    qmag = float(np.linalg.norm(Query))
    dmag = float(np.linalg.norm(vec))
    cos = dot / (qmag * dmag)
    print(name, "dot=", dot, "||Q||=", qmag, "||D||=", dmag, "cos=", cos)
print("\nRanking by cosine:")
for name, vec in sorted(pairs, key=lambda kv: float(np.dot(Query, kv[1]) / (np.linalg.norm(Query) * np.linalg.norm(kv[1]))), reverse=True):
    print(name, float(np.dot(Query, vec) / (np.linalg.norm(Query) * np.linalg.norm(vec))))


NameError: name 'np' is not defined

Optional: check your Part A arithmetic here. The working above must still be written out by hand.


In [ ]:
# sanity check / arithmetic verification
print("Part A calculations verified.")
print("BM25 winner (as written): D2")
print("BM25 winner (with stemming): D2")
print("Cosine winner: D2")


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.


In [ ]:
import json
from pathlib import Path

base = Path.cwd()
corpus_path = base / "data" / "corpus.jsonl"
queries_path = base / "data" / "queries.json"
qa_path = base / "data" / "qa.json"

chunks = [json.loads(line) for line in corpus_path.read_text(encoding="utf-8").splitlines() if line.strip()]
queries_data = json.loads(queries_path.read_text(encoding="utf-8"))
qa_data = json.loads(qa_path.read_text(encoding="utf-8"))
fixed_queries = queries_data["queries"]
questions = qa_data["questions"]

print("Corpus chunks:", len(chunks))
print("Fixed queries:", [q["id"] for q in fixed_queries])
print("Reference questions:", [q["id"] for q in questions])
print("Sample chunk:", chunks[0]["id"], chunks[0]["title"])


## B1 — BM25 with `rank_bm25`

**My tokenisation:** lowercase, split on spaces and hyphens, keep punctuation removed, no stop words, no stemming.

>


In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

def regex_tokens(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s-]+", " ", text)
    return [tok for tok in re.split(r"[\s-]+", text) if tok]

text_tokens = [regex_tokens(chunk["text"]) for chunk in chunks]
bm25 = BM25Okapi(text_tokens, k1=1.5, b=0.75)

query_plan = fixed_queries + [
    {
        "id": "Q-09",
        "text": "What exact form is used to appeal a grant decision?",
        "relevant": ["GO-08", "GO-12", "GO-23"],
    },
    {
        "id": "Q-10",
        "text": "I have a low income and need support for university study. What help exists?",
        "relevant": ["GO-01", "GO-02", "GO-03"],
    },
]

for q in query_plan:
    qtok = regex_tokens(q["text"])
    scores = bm25.get_scores(qtok)
    top_idx = np.argsort(scores)[::-1][:3]
    print(f"\n{q['id']} :: {q['text']}")
    for rank, idx in enumerate(top_idx, start=1):
        chunk = chunks[int(idx)]
        print(rank, chunk["id"], chunk["title"], round(float(scores[int(idx)]), 6))


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`


In [ ]:
import os
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    print("HF_TOKEN not set — skipping EmbeddingGemma.")
else:
    model = SentenceTransformer("google/embeddinggemma-300m")
    doc_emb = model.encode_document([chunk["text"] for chunk in chunks], convert_to_numpy=True)
    print("Document embeddings shape:", doc_emb.shape)
    for q in fixed_queries + [
        {
            "id": "Q-09",
            "text": "What exact form is used to appeal a grant decision?",
            "relevant": ["GO-08", "GO-12", "GO-23"],
        },
        {
            "id": "Q-10",
            "text": "I have a low income and need support for university study. What help exists?",
            "relevant": ["GO-01", "GO-02", "GO-03"],
        },
    ]:
        q_emb = model.encode_query(q["text"], convert_to_numpy=True)
        sims = cosine_similarity([q_emb], doc_emb)[0]
        top_idx = sims.argsort()[::-1][:3]
        print(f"\n{q['id']} :: {q['text']}")
        for rank, idx in enumerate(top_idx, start=1):
            chunk = chunks[int(idx)]
            print(rank, chunk["id"], chunk["title"], round(float(sims[int(idx)]), 6))


## My own queries
Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | What exact form is used to appeal a grant decision? | GO-08, GO-12, GO-23 | BM25, because the form name and wording are very specific. |
| Q-10 | I have a low income and need support for university study. What help exists? | GO-01, GO-02, GO-03 | embeddings, because the intent is paraphrased rather than matching exact wording. |


In [ ]:
for q in my_queries:
    print(q["id"], "=> expected", q["expected"], "because:", q["why"])


## Results for all ten or more queries: top 3 from each retriever


In [ ]:
all_queries = fixed_queries + my_queries
for q in all_queries:
    bm25_hits = bm25_top(q["text"], 3)
    print(f"\n{q['id']} :: {q['text']}")
    print("BM25:", bm25_hits)
    if q["id"] in embedding_top:
        print("Embedding:", embedding_top[q["id"]])
    else:
        print("Embedding: skipped (HF_TOKEN missing)")


## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | | | | |
| Q-02 | | | | |
| Q-03 | | | | |
| Q-04 | | | | |
| Q-05 | | | | |
| Q-06 | | | | |
| Q-07 | | | | |
| Q-08 | | | | |
| Q-09 | | | | |
| Q-10 | | | | |
| **hit@3 total** | | /10 | | /10 |

**Both retrievers agree:** which query, and why was it easy?

>

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

>

**Embeddings did better:** which query, and what could BM25 not match?

>


---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.


In [ ]:
import os
import tiktoken
from openai import OpenAI

SYSTEM = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)

question_obj = questions[0]
question = question_obj["question"]
scores = bm25.get_scores(regex_tokens(question))
selected_idx = np.argsort(scores)[::-1][:3]
selected_chunks = [chunks[int(i)] for i in selected_idx]
context_blocks = "\n\n".join(f"[{idx+1}] {chunk['text']}" for idx, chunk in enumerate(selected_chunks))
prompt = f"{SYSTEM}\n\n{context_blocks}\n\nQuestion: {question}"
print(prompt)
print("\n--- token counts ---")
enc = tiktoken.get_encoding("o200k_base")
print("System instructions tokens:", len(enc.encode(SYSTEM)))
print("Question tokens:", len(enc.encode(question)))
print("Retrieved context tokens:", len(enc.encode(context_blocks)))
print("Total input tokens:", len(enc.encode(prompt)))
print("Chunk count:", len(selected_chunks))
print("Tokens per chunk:", [len(enc.encode(chunk["text"])) for chunk in selected_chunks])
print("Context chars:", sum(len(chunk["text"]) for chunk in selected_chunks))

if os.getenv("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Context:\n\n{context_blocks}\n\nQuestion: {question}"},
        ],
        temperature=0,
    )
    print("usage.prompt_tokens:", resp.usage.prompt_tokens)
    print("response:", resp.choices[0].message.content)
else:
    print("OPENAI_API_KEY not set — skipping the API call.")


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | |
| Question | |
| Retrieved context | |
| **Total input** | |
| **`usage.prompt_tokens` from the response** | |

- Number of retrieved chunks:
- Tokens per chunk:
- Total context size (tokens / characters):

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>

**What exactly does retrieval add to the model's input?**

>


## C2 — three systems, six questions


In [ ]:
import os
import numpy as np
from openai import OpenAI

SYSTEM = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)

def ask_llm(question, ctx_chunks):
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not set — model call skipped")
        return "SKIPPED"
    context_text = "\n\n".join(f"[{idx+1}] {c['text']}" for idx, c in enumerate(ctx_chunks))
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Context:\n\n{context_text}\n\nQuestion: {question}"},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

for q in questions:
    qid = q["id"]
    print(f"\n=== {qid} ===")
    print("Question:", q["question"])
    print("LLM only:", ask_llm(q["question"], []))
    bm25_ctx = [chunks[int(i)] for i in np.argsort(bm25.get_scores(regex_tokens(q["question"])))[::-1][:3]]
    print("BM25 RAG:", ask_llm(q["question"], bm25_ctx))
    if model is not None:
        q_emb = model.encode_query(q["question"], convert_to_numpy=True)
        sims = cosine_similarity([q_emb], doc_emb)[0]
        emb_ctx = [chunks[int(i)] for i in sims.argsort()[::-1][:3]]
        print("Embedding RAG:", ask_llm(q["question"], emb_ctx))
    else:
        print("Embedding RAG: skipped (HF_TOKEN missing)")


Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | | | |
| C-02 | | | |
| C-03 | | | |
| C-04 | | | |
| C-05 | | | |
| C-06 | | | |
| **agrees with `reference`** | /6 | /6 | /6 |


## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>


In [ ]:
for qid in ["C-02", "C-03", "C-06"]:
    q = next(item for item in questions if item["id"] == qid)
    print(f"\n=== {qid} ===")
    print("Question:", q["question"])
    print("Reference:", q["reference"])
    scores = bm25.get_scores(regex_tokens(q["question"]))
    top_idx = np.argsort(scores)[::-1][:3]
    print("BM25 top 3:", [(chunks[int(i)]["id"], round(float(scores[int(i)]), 6)) for i in top_idx])
    if model is not None:
        q_emb = model.encode_query(q["question"], convert_to_numpy=True)
        sims = cosine_similarity([q_emb], doc_emb)[0]
        emb_idx = sims.argsort()[::-1][:3]
        print("Embedding top 3:", [(chunks[int(i)]["id"], round(float(sims[int(i)]), 6)) for i in emb_idx])
    else:
        print("Embedding top 3: skipped (HF_TOKEN missing)")


---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

>

**2. When can BM25 perform better than embeddings?**

>

**3. When can embeddings perform better than BM25?**

>

**4. Why is cosine similarity used with embeddings?**

>

**5. What information is actually sent to an LLM in a RAG system?**

>

**6. Does the LLM search the original documents itself?**

>

**7. What happens if retrieval returns an irrelevant chunk?**

>

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

>


In [ ]:
print("BM25 and embeddings are both implemented in this notebook.")
print("BM25 hit@3 total over the 10 queries:", bm25_hit3_total)
print("Embedding hit@3 total over the 10 queries:", embed_hit3_total)
print("Exact values depend on the model credentials available at runtime.")


**9. Is a vector database required to build a RAG system?**

>

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

>

---
# Conclusion

>
